<a href="https://colab.research.google.com/github/edsilva14/infovis/blob/main/data360_exploration_personal_avanzado_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploración Profunda: API Data360 del Banco Mundial

Este cuaderno profundiza en la estructura de la API **Data360**, analizando los metadatos disponibles: cobertura temporal, áreas geográficas y categorías de datos (temas).

**API Endpoint**: `https://data360api.worldbank.org`  
**Documentación**: Data360 Open API Spec

In [7]:
import requests
import pandas as pd
import duckdb
import altair as alt
import json
from sklearn.impute import SimpleImputer  # FIX: movido fuera del comentario
import plotly.graph_objects as go
import numpy as np
import plotly.express as px

# Configuración base
BASE_URL = "https://data360api.worldbank.org/data360"
# alt.renderers.enable('colab')  # Descomentar si se usa en Google Colab


Primero exploraremos las bases de datos disponibles en la API del Banco Mundial, sus indicadores y como se estructuran

In [4]:
import requests
import pandas as pd

BASE_URL_WB = "https://api.worldbank.org/v2"  # FIX: renombrado para no pisar BASE_URL de Data360

print("1. Ver todas las bases de datos disponibles")
r = requests.get(f"{BASE_URL_WB}/sources?format=json&per_page=100")
print("Status:", r.status_code)
data = r.json()
fuentes = data[1]  # la API WB devuelve [metadata, datos]
df_fuentes = pd.DataFrame(fuentes)[["id", "name"]]
print(df_fuentes)
print("2. Ver indicadores disponibles (con búsqueda por tema)")
r = requests.get(f"{BASE_URL_WB}/indicator?format=json&per_page=50&source=2")
data = r.json()
indicadores = data[1]
df_ind = pd.DataFrame(indicadores)[["id", "name"]]
print(df_ind)
print("3. Traer datos concretos (población ARG, BRA, CHL,etc)")
url = "https://api.worldbank.org/v2/country/ARG;BRA;CHL/indicator/SP.POP.TOTL"

r = requests.get(url, params={
    "format": "json",
    "per_page": "500",
    "date": "2000:2023"
})

print("Status:", r.status_code)
print("URL consultada:", r.url)  # para ver exactamente qué se pidió
print("Respuesta cruda:", r.text[:300])

1. Ver todas las bases de datos disponibles
Status: 200
    id                                        name
0    1                              Doing Business
1    2                World Development Indicators
2    3             Worldwide Governance Indicators
3    5           Subnational Malnutrition Database
4    6               International Debt Statistics
..  ..                                         ...
66  89  Identification for Development (ID4D) Data
67  90                                    ICP 2021
68  91                                  PEFA_CRPFM
69  92                   Disability Data Hub (DDH)
70  93                         FPN Datahub Archive

[71 rows x 2 columns]
2. Ver indicadores disponibles (con búsqueda por tema)
                      id                                               name
0      AG.CON.FERT.PT.ZS  Fertilizer consumption (% of fertilizer produc...
1         AG.CON.FERT.ZS  Fertilizer consumption (kilograms per hectare ...
2         AG.LND.AGRI.K2  

La API del Banco Mundial cuenta con un total de 71 bases de datos distintas.
La más completas de estas es la World Development Indicators (WDI, id=2) con más de 1.400 indicadores. Cada dato se estructura en 4 dimensiones:

País — Código ISO2 (AR) e ISO3 (ARG).

Tiempo — Año (series desde 1960 hasta 2024).

Indicador —Código único (ej: SP.POP.TOTL) + nombre descriptivo.

Valor — Número con unidad (%, USD, km², etc.)

ANÁLISIS PBI PER CAPITA DE ARGENTINA

**COMPARO DESDE 1960 HASTA 2024 VS OTROS PAÍSES DE LATINOAMERICA**

Por lo general se suele ver al crecimiento como una forma de bienestar, en donde si el PIB es más grande las personas deberían ser más ricas. Pero esto, ¿es así?

En los siguientes gráficos, analizaremos la desigualdad y su relación con el crecimiento económico.

In [5]:
# Traer todos los países de la región "Latin America & Caribbean"
r = requests.get(f"{BASE_URL_WB}/country?format=json&per_page=300&region=LCN")  # FIX: usa BASE_URL_WB
data = r.json()
paises = data[1]
df_paises = pd.DataFrame(paises)
print(df_paises[["id", "name"]].to_string())

import time

# Códigos de países latinoamericanos más relevantes
paises_latam = [
    "ARG","BRA","CHL","COL","MEX","PER","URY",
    "BOL","ECU","PRY","VEN","GTM","CRI","PAN",
    "DOM","HND","SLV","NIC","CUB","HTI"
]

paises_str = ";".join(paises_latam)

url = f"{BASE_URL_WB}/country/{paises_str}/indicator/SI.POV.GINI"  # FIX: usa BASE_URL_WB
r = requests.get(url, params={
    "format": "json",
    "per_page": "2000",
    "date": "1960:2023"
})

datos = r.json()[1]
df = pd.DataFrame(datos)

# Limpiar y estructurar
df["pais"] = df["country"].apply(lambda x: x["value"])
df["iso"]  = df["countryiso3code"]
df = df[["pais", "iso", "date", "value"]].dropna()
df.columns = ["País", "ISO", "Año", "PBI_per_capita"]
df["Año"] = df["Año"].astype(int)

print(df.shape)
print(df.head())

# Columna para destacar ARG vs el resto
df["color"] = df["ISO"].apply(lambda x: "Argentina" if x == "ARG" else "Resto LATAM")

print(df["País"].unique())  # verificar que estén todos los países

import altair as alt

# Líneas del resto de LATAM (grises, finas)
lineas_latam = alt.Chart(df[df["color"] == "Resto LATAM"]).mark_line(
    opacity=0.3,
    strokeWidth=1
).encode(
    x=alt.X("Año:O", title="Año"),
    y=alt.Y("PBI_per_capita:Q", title="Indice de Gini"),
    color=alt.value("gray"),
    detail="País:N",
    tooltip=["País", "Año", "PBI_per_capita"]
)

# Línea de Argentina (destacada en azul)
linea_arg = alt.Chart(df[df["color"] == "Argentina"]).mark_line(
    strokeWidth=3,
    color="#1f77b4"
).encode(
    x="Año:O",
    y="PBI_per_capita:Q",
    tooltip=["País", "Año", "PBI_per_capita"]
)

# Etiqueta final sobre Argentina
label_arg = alt.Chart(df[(df["color"] == "Argentina") & (df["Año"] == df["Año"].max())]).mark_text(
    align="left", dx=5, fontWeight="bold", color="#1f77b4"
).encode(
    x="Año:O",
    y="PBI_per_capita:Q",
    text="País:N"
)

# Combinar todo
grafico = (lineas_latam + linea_arg + label_arg).properties(
    title="Nivel de Desigualdad en la región",
    width=800,
    height=450
).interactive()  # ← zoom y pan con el mouse

grafico

     id                            name
0   ABW                           Aruba
1   ARG                       Argentina
2   ATG             Antigua and Barbuda
3   BHS                    Bahamas, The
4   BLZ                          Belize
5   BOL                         Bolivia
6   BRA                          Brazil
7   BRB                        Barbados
8   CHL                           Chile
9   COL                        Colombia
10  CRI                      Costa Rica
11  CUB                            Cuba
12  CUW                         Curacao
13  CYM                  Cayman Islands
14  DMA                        Dominica
15  DOM              Dominican Republic
16  ECU                         Ecuador
17  GRD                         Grenada
18  GTM                       Guatemala
19  GUY                          Guyana
20  HND                        Honduras
21  HTI                           Haiti
22  JAM                         Jamaica
23  KNA             St. Kitts and Nevis


alt.LayerChart(...)

Podemos ver una tendencia a lo largo de la historia que muestra una reducción de los niveles de desigualdad, es decir, los países cada vez tienden a ser menos regresivos en América Latina como producto de intervención estatal en estos asuntos.

Asimismo, se aprecia un ligero aumento en los coeficientes de Gini en el período post-pandemia.

In [10]:
"""
Mapa Interactivo: PBI per Cápita en América Latina (1960–2023)
=============================================================
Fuente de datos : World Bank API v2 (NY.GDP.PCAP.CD)
Librería de viz : Plotly Express (choropleth + slider de años)
Cómo correrlo   : python mapa_pbi_latam.py
                  → abre el mapa en el browser automáticamente
                  → también funciona en Jupyter/Colab (última línea)
"""


# ── 1. Países de América Latina (ISO3) ───────────────────────────────────────
PAISES_LATAM = [
    "ARG", "BOL", "BRA", "CHL", "COL", "CRI", "CUB", "DOM", "ECU",
    "SLV", "GTM", "HTI", "HND", "MEX", "NIC", "PAN", "PRY", "PER",
    "URY", "VEN",
]

# ── 2. Descarga de datos vía World Bank API ───────────────────────────────────
def fetch_gdp_per_capita(paises: list[str], anio_desde: int = 1990, anio_hasta: int = 2023) -> pd.DataFrame:
    """
    Descarga NY.GDP.PCAP.CD (PBI per cápita en USD corrientes)
    para los países y rango de años indicados.
    """
    paises_str = ";".join(paises)
    url = f"https://api.worldbank.org/v2/country/{paises_str}/indicator/SI.POV.GINI"

    params = {
        "format":   "json",
        "per_page": "2000",
        "date":     f"{anio_desde}:{anio_hasta}",
    }

    print(f"Descargando datos del Banco Mundial ({anio_desde}–{anio_hasta})...")
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    # La API WB devuelve [metadata, lista_de_datos]
    registros = data[1]
    df = pd.DataFrame(registros)

    # Extraer nombre del país desde la columna anidada
    df["pais"]  = df["country"].apply(lambda x: x["value"] if isinstance(x, dict) else None)
    df["iso3"]  = df["countryiso3code"]
    df["anio"]  = df["date"].astype(int)
    df["pbi_pc"] = pd.to_numeric(df["value"], errors="coerce")

    # Quedarse solo con columnas útiles y filas con dato
    df = (
        df[["pais", "iso3", "anio", "pbi_pc"]]
        .dropna(subset=["pbi_pc"])
        .sort_values(["iso3", "anio"])
        .reset_index(drop=True)
    )

    print(f"  ✓ {len(df)} observaciones cargadas para {df['iso3'].nunique()} países.")
    return df


# ── 3. Construcción del mapa ──────────────────────────────────────────────────
def construir_mapa(df: pd.DataFrame) -> px.choropleth:
    """
    Choropleth animado con slider por año.
    Escala de color divergente centrada en la mediana regional.
    """
    # Rango de color: percentil 5–95 para no distorsionar por outliers
    vmin = df["pbi_pc"].quantile(0.05)
    vmax = df["pbi_pc"].quantile(0.95)

    fig = px.choropleth(
        df,
        locations="iso3",                    # columna con código ISO3
        color="pbi_pc",                      # variable a colorear
        hover_name="pais",                   # tooltip: nombre del país
        hover_data={
            "iso3":   False,
            "anio":   True,
            "pbi_pc": ":,.0f",               # formato con separador de miles
        },
        animation_frame="anio",              # slider de años
        color_continuous_scale="RdYlGn",     # rojo (bajo) → amarillo → verde (alto)
        range_color=(vmin, vmax),
       scope="south america",               # zoom inicial en Sudamérica
        labels={
            "pbi_pc": "PBI per cápita (USD)",
            "anio":   "Año",
        },
        title="Grado de desigualdad en América Latina",
    )

    # ── Ajustes de layout ────────────────────────────────────────────────────
    fig.update_layout(
        title={
            "text":  "Grado de desigualdad en América Latina",
            "x":     0.5,
            "xanchor": "center",
            "font":  {"size": 20},
        },
        geo=dict(
            showframe=False,
            showcoastlines=True,
            coastlinecolor="white",
            showland=True,
            landcolor="#f5f5f5",
            showocean=True,
            oceancolor="#cce5f5",
            showlakes=True,
            lakecolor="#cce5f5",
            showcountries=True,
            countrycolor="white",
            # Centrar en LATAM incluyendo Caribe y México
            projection_type="natural earth",
            lataxis_range=[-60, 35],
            lonaxis_range=[-120, -30],
        ),
        coloraxis_colorbar=dict(
            title="USD corrientes",
            tickformat=".0f",
            len=0.75,
        ),
        # Slider de años más prolijo
        sliders=[{
            "currentvalue": {
                "prefix": "Año: ",
                "font": {"size": 14, "color": "#333"},
            },
            "pad": {"t": 50},
        }],
        margin={"r": 0, "t": 80, "l": 0, "b": 0},
        height=650,
        paper_bgcolor="#ffffff",
    )

    # Ralentizar la animación para que sea más legible
    fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 400
    fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 300

    return fig


# ── 4. Main ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Descarga (podés ajustar el rango de años)
    df = fetch_gdp_per_capita(PAISES_LATAM, anio_desde=1990, anio_hasta=2023)

    # Vista rápida de los datos
    print("\nEjemplo de datos:")
    print(df.head(10).to_string(index=False))
    print(f"\nAños disponibles: {sorted(df['anio'].unique())}")

    # Construir y mostrar el mapa
    fig = construir_mapa(df)

    # → En script .py: abre el browser
    fig.show()

    # → En Jupyter / Colab: descomentar la línea de abajo
    # fig.show(renderer="colab")

    # → Exportar a HTML standalone (sin internet requerido para verlo)
    fig.write_html("mapa_pbi_latam.html", include_plotlyjs="cdn")
    print("\n✓ Mapa exportado como 'mapa_pbi_latam.html'")

Descargando datos del Banco Mundial (1990–2023)...
  ✓ 425 observaciones cargadas para 19 países.

Ejemplo de datos:
     pais iso3  anio  pbi_pc
Argentina  ARG  1991    46.8
Argentina  ARG  1992    45.5
Argentina  ARG  1993    44.8
Argentina  ARG  1994    45.9
Argentina  ARG  1995    48.9
Argentina  ARG  1996    49.5
Argentina  ARG  1997    49.1
Argentina  ARG  1998    50.7
Argentina  ARG  1999    49.8
Argentina  ARG  2000    51.0

Años disponibles: [np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]



✓ Mapa exportado como 'mapa_pbi_latam.html'


Haciendo foco en la región, se puede ver una progresiva mejora en los niveles de distribución del ingreso en los países de Argentina, Uruguay, Bolivia y Perú; producto de políticas económicas más intervencionistas y de mayor foco hacia una distribución equitativa del ingreso.

In [9]:
"""
Scatter Plot Interactivo: Índice de Gini vs PBI per Cápita
==========================================================
Fuente de datos : World Bank API v2
  - SI.POV.GINI   → Índice de Gini
  - NY.GDP.PCAP.CD → PBI per cápita (USD corrientes)
Librería de viz : Plotly (scatter animado + filtro por país)
Cómo correrlo   : python scatter_gini_pbi.py
"""


# ── 1. Países y parámetros ────────────────────────────────────────────────────
PAISES_LATAM = [
    "ARG", "BOL", "BRA", "CHL", "COL", "CRI", "DOM", "ECU",
    "SLV", "GTM", "HTI", "HND", "MEX", "NIC", "PAN", "PRY",
    "PER", "URY", "VEN",
]

# Colores por país (ARG destacado)
COLOR_ARG     = "#1f77b4"
COLOR_RESTO   = "#b0b0b0"
COLOR_HIGHLIGHT = {
    "ARG": "#1f77b4",
    "BRA": "#e74c3c",
    "CHL": "#2ecc71",
    "MEX": "#f39c12",
    "COL": "#9b59b6",
    "URY": "#00b4d8",
    "PER": "#e67e22",
    "BOL": "#1abc9c",
}

ANIO_DESDE = 1990
ANIO_HASTA = 2023


# ── 2. Descarga de datos ──────────────────────────────────────────────────────
def fetch_indicator(paises: list[str], indicator: str, anio_desde: int, anio_hasta: int) -> pd.DataFrame:
    paises_str = ";".join(paises)
    url = f"https://api.worldbank.org/v2/country/{paises_str}/indicator/{indicator}"
    params = {"format": "json", "per_page": "2000", "date": f"{anio_desde}:{anio_hasta}"}

    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    if not isinstance(data, list) or len(data) < 2 or not data[1]:
        return pd.DataFrame()

    df = pd.DataFrame(data[1])
    df["pais"]  = df["country"].apply(lambda x: x["value"] if isinstance(x, dict) else None)
    df["iso3"]  = df["countryiso3code"]
    df["anio"]  = pd.to_numeric(df["date"], errors="coerce")
    df["valor"] = pd.to_numeric(df["value"], errors="coerce")
    return df[["pais", "iso3", "anio", "valor"]].dropna()


def fetch_population(paises: list[str], anio_desde: int, anio_hasta: int) -> pd.DataFrame:
    return fetch_indicator(paises, "SP.POP.TOTL", anio_desde, anio_hasta).rename(columns={"valor": "poblacion"})


def build_dataset(paises: list[str], anio_desde: int, anio_hasta: int) -> pd.DataFrame:
    print("Descargando Índice de Gini...")
    df_gini = fetch_indicator(paises, "SI.POV.GINI", anio_desde, anio_hasta).rename(columns={"valor": "gini"})

    print("Descargando PBI per cápita...")
    df_pbi  = fetch_indicator(paises, "NY.GDP.PCAP.CD", anio_desde, anio_hasta).rename(columns={"valor": "pbi_pc"})

    print("Descargando población (para tamaño de burbuja)...")
    df_pop  = fetch_population(paises, anio_desde, anio_hasta)

    # Merge por país y año
    df = (
        df_gini
        .merge(df_pbi[["iso3", "anio", "pbi_pc"]], on=["iso3", "anio"], how="inner")
        .merge(df_pop[["iso3", "anio", "poblacion"]], on=["iso3", "anio"], how="left")
    )

    # El Gini tiene datos esparsos → forward fill por país para el slider
    df = df.sort_values(["iso3", "anio"])
    df["gini"]     = df.groupby("iso3")["gini"].transform(lambda s: s.ffill().bfill())
    df["pbi_pc"]   = df.groupby("iso3")["pbi_pc"].transform(lambda s: s.ffill().bfill())
    df["poblacion"] = df.groupby("iso3")["poblacion"].transform(lambda s: s.ffill().bfill())

    df = df.dropna(subset=["gini", "pbi_pc"])
    df["anio"] = df["anio"].astype(int)

    # Tamaño de burbuja normalizado
    df["bubble_size"] = (df["poblacion"] / df["poblacion"].max() * 60 + 8).fillna(10).round(1)

    print(f"  ✓ Dataset listo: {len(df)} filas · {df['iso3'].nunique()} países · años {df['anio'].min()}–{df['anio'].max()}")
    return df.reset_index(drop=True)


# ── 3. Construcción del scatter interactivo ───────────────────────────────────
def construir_scatter(df: pd.DataFrame) -> go.Figure:
    """
    Scatter Gini vs PBI per cápita con:
      - Slider de años (animación)
      - Dropdown para filtrar por país (resalta el seleccionado)
      - Burbujas proporcionales a la población
      - Tooltip detallado
    """
    anios  = sorted(df["anio"].unique())
    paises = sorted(df["pais"].unique())

    def color_for(iso3):
        return COLOR_HIGHLIGHT.get(iso3, COLOR_RESTO)

    # ── Frames (uno por año) ──────────────────────────────────────────────────
    frames = []
    for anio in anios:
        df_a = df[df["anio"] == anio]

        scatter = go.Scatter(
            x=df_a["pbi_pc"],
            y=df_a["gini"],
            mode="markers+text",
            marker=dict(
                size=df_a["bubble_size"],
                color=[color_for(iso) for iso in df_a["iso3"]],
                opacity=0.85,
                line=dict(width=1, color="white"),
            ),
            text=df_a["iso3"],
            textposition="top center",
            textfont=dict(size=9),
            customdata=np.stack([
                df_a["pais"],
                df_a["gini"].round(1),
                df_a["pbi_pc"].round(0),
                (df_a["poblacion"] / 1e6).round(2),
            ], axis=-1),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Gini: %{customdata[1]}<br>"
                "PBI per cápita: USD %{customdata[2]:,.0f}<br>"
                "Población: %{customdata[3]:.1f} M<br>"
                "<extra></extra>"
            ),
        )
        frames.append(go.Frame(data=[scatter], name=str(anio)))

    # ── Trace inicial (primer año disponible) ─────────────────────────────────
    df_init = df[df["anio"] == anios[0]]
    trace_init = go.Scatter(
        x=df_init["pbi_pc"],
        y=df_init["gini"],
        mode="markers+text",
        marker=dict(
            size=df_init["bubble_size"],
            color=[color_for(iso) for iso in df_init["iso3"]],
            opacity=0.85,
            line=dict(width=1, color="white"),
        ),
        text=df_init["iso3"],
        textposition="top center",
        textfont=dict(size=9),
        customdata=np.stack([
            df_init["pais"],
            df_init["gini"].round(1),
            df_init["pbi_pc"].round(0),
            (df_init["poblacion"] / 1e6).round(2),
        ], axis=-1),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Gini: %{customdata[1]}<br>"
            "PBI per cápita: USD %{customdata[2]:,.0f}<br>"
            "Población: %{customdata[3]:.1f} M<br>"
            "<extra></extra>"
        ),
    )

    fig = go.Figure(data=[trace_init], frames=frames)

    # ── Botones del dropdown (filtrar por país) ───────────────────────────────
    # "Todos" = restaura opacidad y tamaño originales
    dropdown_buttons = [
        dict(
            label="Todos los países",
            method="restyle",
            args=[{
                "marker.opacity": [0.85],
                "marker.color":   [[color_for(iso) for iso in df_init["iso3"]]],
            }]
        )
    ]

    for pais_nombre in paises:
        iso_sel = df.loc[df["pais"] == pais_nombre, "iso3"].iloc[0]

        # País seleccionado: color destacado / resto: gris muy transparente
        colors_sel  = [
            (COLOR_HIGHLIGHT.get(iso_sel, COLOR_ARG) if iso == iso_sel else "#d0d0d0")
            for iso in df_init["iso3"]
        ]
        opacity_sel = [
            (1.0 if iso == iso_sel else 0.15)
            for iso in df_init["iso3"]
        ]
        size_sel = [
            (df_init.loc[df_init["iso3"] == iso, "bubble_size"].values[0] * 1.6
             if iso == iso_sel
             else df_init.loc[df_init["iso3"] == iso, "bubble_size"].values[0])
            for iso in df_init["iso3"]
        ]

        dropdown_buttons.append(dict(
            label=pais_nombre,
            method="restyle",
            args=[{
                "marker.color":   [colors_sel],
                "marker.opacity": [opacity_sel],
                "marker.size":    [size_sel],
            }]
        ))

    # ── Slider de años ────────────────────────────────────────────────────────
    sliders = [{
        "steps": [
            {
                "args": [[str(a)], {"frame": {"duration": 400}, "mode": "immediate",
                                    "transition": {"duration": 300}}],
                "label": str(a),
                "method": "animate",
            }
            for a in anios
        ],
        "currentvalue": {
            "prefix": "Año: ",
            "font": {"size": 14, "color": "#333"},
        },
        "pad": {"t": 55, "b": 10},
        "len": 0.9,
        "x": 0.05,
    }]

    # ── Play / Pause ──────────────────────────────────────────────────────────
    play_pause = [{
        "type": "buttons",
        "showactive": False,
        "x": 0.05, "y": -0.07,
        "xanchor": "left",
        "buttons": [
            {"label": "▶ Play",  "method": "animate",
             "args": [None, {"frame": {"duration": 600}, "fromcurrent": True,
                             "transition": {"duration": 400}}]},
            {"label": "⏸ Pausa", "method": "animate",
             "args": [[None], {"frame": {"duration": 0}, "mode": "immediate",
                                "transition": {"duration": 0}}]},
        ],
    }]

    # ── Layout ────────────────────────────────────────────────────────────────
    fig.update_layout(
        title={
            "text":    "Índice de Gini vs PBI per Cápita — América Latina",
            "x":       0.5,
            "xanchor": "center",
            "font":    {"size": 20},
        },
        xaxis=dict(
            title="PBI per cápita (USD corrientes, escala log)",
            type="log",
            tickformat="$,.0f",
            gridcolor="#eeeeee",
            showline=True,
            linecolor="#cccccc",
        ),
        yaxis=dict(
            title="Índice de Gini (0 = igualdad total · 100 = desigualdad máxima)",
            range=[25, 65],
            gridcolor="#eeeeee",
            showline=True,
            linecolor="#cccccc",
        ),
        plot_bgcolor="#fafafa",
        paper_bgcolor="#ffffff",
        hovermode="closest",
        height=650,
        margin={"t": 90, "b": 120, "l": 70, "r": 40},
        sliders=sliders,
        updatemenus=[
            # Dropdown filtro de país
            dict(
                buttons=dropdown_buttons,
                direction="down",
                showactive=True,
                x=0.80, xanchor="left",
                y=1.13, yanchor="top",
                bgcolor="#f0f0f0",
                bordercolor="#cccccc",
                font={"size": 12},
            ),
            # Play / Pause
            *play_pause,
        ],
        annotations=[
            dict(
                text="Filtrar país:",
                x=0.80, xref="paper",
                y=1.16, yref="paper",
                showarrow=False,
                font={"size": 12, "color": "#555"},
            ),
            dict(
                text="Tamaño de burbuja proporcional a la población · Fuente: World Bank API",
                x=0.5, xref="paper",
                y=-0.18, yref="paper",
                showarrow=False,
                font={"size": 10, "color": "#999"},
            ),
        ],
    )

    return fig


# ── 4. Main ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df  = build_dataset(PAISES_LATAM, ANIO_DESDE, ANIO_HASTA)

    print("\nEjemplo de datos:")
    print(df.head(10).to_string(index=False))

    fig = construir_scatter(df)

    # → Script .py: abre el browser
    fig.show()

    # → Jupyter / Colab: descomentar
    # fig.show(renderer="colab")

    # → Exportar HTML standalone
    fig.write_html("scatter_gini_pbi.html", include_plotlyjs="cdn")
    print("\n✓ Exportado como 'scatter_gini_pbi.html'")

Descargando Índice de Gini...
Descargando PBI per cápita...
Descargando población (para tamaño de burbuja)...
  ✓ Dataset listo: 425 filas · 19 países · años 1990–2023

Ejemplo de datos:
     pais iso3  anio  gini      pbi_pc  poblacion  bubble_size
Argentina  ARG  1991  46.8 5709.247841   33230294         17.4
Argentina  ARG  1992  45.5 6789.996111   33693527         17.6
Argentina  ARG  1993  44.8 6931.855964   34152717         17.7
Argentina  ARG  1994  45.9 7437.562423   34613491         17.8
Argentina  ARG  1995  48.9 7357.616277   35070020         18.0
Argentina  ARG  1996  49.5 7663.212713   35513793         18.1
Argentina  ARG  1997  49.1 8146.787100   35947791         18.2
Argentina  ARG  1998  50.7 8218.992128   36372860         18.3
Argentina  ARG  1999  49.8 7705.542883   36794682         18.5
Argentina  ARG  2000  51.0 7637.014892   37213984         18.6



✓ Exportado como 'scatter_gini_pbi.html'


Por último, analizamos la relación entre el crecimiento económico y los niveles del indice de Gini en dichas economías

Procedemos a analizar como cambio el PBI per capita de Argentina vs el promedio de latinoamerica año tras año

In [4]:
# Promedio regional por año (excluyendo Argentina)
df_latam_avg = (
    df[df["ISO"] != "ARG"]
    .groupby("Año")["PBI_per_capita"]
    .mean()
    .reset_index()
)
df_latam_avg["País"] = "Promedio LATAM"
df_latam_avg.columns = ["Año", "PBI_per_capita", "País"]

# Datos solo de Argentina
df_arg = df[df["ISO"] == "ARG"][["Año", "PBI_per_capita", "País"]].copy()

# Unir ambos
df_barras = pd.concat([df_arg, df_latam_avg], ignore_index=True)
print(df_barras.head(10))
barras = alt.Chart(df_barras).mark_bar().encode(
    x=alt.X("Año:O",
            title="Año",
            axis=alt.Axis(labelAngle=-45, values=list(range(1960, 2024, 5)))
    ),
    y=alt.Y("PBI_per_capita:Q",
            title="PBI per cápita (USD corrientes)"
    ),
    color=alt.Color("País:N",
        scale=alt.Scale(
            domain=["Argentina", "Promedio LATAM"],
            range=["#1f77b4", "#aec7e8"]  # azul oscuro vs azul claro
        ),
        legend=alt.Legend(title="")
    ),
    xOffset="País:N",  # ← esto pone las barras LADO A LADO por año
    tooltip=[
        alt.Tooltip("País:N", title="País"),
        alt.Tooltip("Año:O", title="Año"),
        alt.Tooltip("PBI_per_capita:Q", title="PBI per cápita (USD)", format=",.0f")
    ]
).properties(
    title="PBI per cápita: Argentina vs Promedio Latinoamérica (1960–2023)",
    width=900,
    height=450
).interactive()

barras
# Calcular diferencia
df_pivot = df_barras.pivot(index="Año", columns="País", values="PBI_per_capita").reset_index()
df_pivot["Diferencia"] = df_pivot["Argentina"] - df_pivot["Promedio LATAM"]
df_pivot["Color"] = df_pivot["Diferencia"].apply(lambda x: "Por encima" if x >= 0 else "Por debajo")

barras_dif = alt.Chart(df_pivot).mark_bar().encode(
    x=alt.X("Año:O", axis=alt.Axis(labelAngle=-45, values=list(range(1960, 2024, 5)))),
    y=alt.Y("Diferencia:Q", title="Diferencia vs promedio LATAM (USD)"),
    color=alt.Color("Color:N",
        scale=alt.Scale(
            domain=["Por encima", "Por debajo"],
            range=["#2ecc71", "#e74c3c"]  # verde / rojo
        ),
        legend=alt.Legend(title="Argentina")
    ),
    tooltip=[
        alt.Tooltip("Año:O", title="Año"),
        alt.Tooltip("Diferencia:Q", title="Diferencia (USD)", format=",.0f")
    ]
).properties(
    title="¿Cuánto supera (o no) Argentina al promedio regional? (1960–2023)",
    width=900,
    height=350
).interactive()

barras_dif

    Año  PBI_per_capita       País
0  2023    14261.846567  Argentina
1  2022    13962.189409  Argentina
2  2021    10738.017922  Argentina
3  2020     8535.599380  Argentina
4  2019     9955.974787  Argentina
5  2018    11752.799892  Argentina
6  2017    14532.500931  Argentina
7  2016    12699.962314  Argentina
8  2015    13679.626498  Argentina
9  2014    12233.144412  Argentina


alt.Chart(...)

Analizamos como varía el PBI per capita de argentina en comparación con sus países limitrofes

In [5]:
# FIX: df_sel no estaba definida. Se filtra df con los países limítrofes de Argentina.
paises_limitrofes = ["ARG", "BRA", "CHL", "URY", "BOL", "PRY"]
df_sel = df[df["ISO"].isin(paises_limitrofes)].copy()

df_promedio = (
    df_sel[df_sel["ISO"] != "ARG"]
    .groupby("Año")["PBI_per_capita"]
    .mean()
    .reset_index()
)
df_promedio["País"] = "Promedio regional"
df_promedio["ISO"]  = "AVG"

df_sel_con_avg = pd.concat([df_sel, df_promedio], ignore_index=True)

colores = {
    "Argentina":         "#1f77b4",
    "Uruguay":           "#2ecc71",
    "Chile":             "#e74c3c",
    "Brazil":            "#f39c12",
    "Bolivia":           "#9b59b6",
    "Paraguay":          "#00b4d8",
    "Promedio regional": "#2d2d2d"
}

df_labels = df_sel_con_avg.loc[df_sel_con_avg.groupby("País")["Año"].idxmax()]

# ── Separar capas por tipo ─────────────────────────────────────

# Países normales (línea sólida, grosor normal)
df_paises = df_sel_con_avg[~df_sel_con_avg["País"].isin(["Argentina", "Promedio regional"])]
lineas_paises = alt.Chart(df_paises).mark_line(strokeWidth=1.8).encode(
    x=alt.X("Año:O", title="Año",
            axis=alt.Axis(labelAngle=-45, values=list(range(1960, 2024, 5)))),
    y=alt.Y("PBI_per_capita:Q", title="PBI per cápita (USD corrientes)",
            axis=alt.Axis(format="$,.0f")),
    color=alt.Color("País:N",
        scale=alt.Scale(domain=list(colores.keys()), range=list(colores.values())),
        legend=alt.Legend(title="País")),
    tooltip=[
        alt.Tooltip("País:N",            title="País"),
        alt.Tooltip("Año:O",             title="Año"),
        alt.Tooltip("PBI_per_capita:Q",  title="PBI per cápita (USD)", format="$,.0f")
    ]
)

# Argentina (línea sólida, más gruesa)
df_arg_sel = df_sel_con_avg[df_sel_con_avg["País"] == "Argentina"]
linea_arg = alt.Chart(df_arg_sel).mark_line(strokeWidth=3.5).encode(
    x="Año:O",
    y="PBI_per_capita:Q",
    color=alt.Color("País:N",
        scale=alt.Scale(domain=list(colores.keys()), range=list(colores.values()))),
    tooltip=[
        alt.Tooltip("País:N",            title="País"),
        alt.Tooltip("Año:O",             title="Año"),
        alt.Tooltip("PBI_per_capita:Q",  title="PBI per cápita (USD)", format="$,.0f")
    ]
)

# Promedio regional (línea punteada)
df_avg_sel = df_sel_con_avg[df_sel_con_avg["País"] == "Promedio regional"]
linea_avg = alt.Chart(df_avg_sel).mark_line(
    strokeWidth=2.5,
    strokeDash=[6, 3]
).encode(
    x="Año:O",
    y="PBI_per_capita:Q",
    color=alt.Color("País:N",
        scale=alt.Scale(domain=list(colores.keys()), range=list(colores.values()))),
    tooltip=[
        alt.Tooltip("País:N",            title="País"),
        alt.Tooltip("Año:O",             title="Año"),
        alt.Tooltip("PBI_per_capita:Q",  title="PBI per cápita (USD)", format="$,.0f")
    ]
)

# ── Puntos ─────────────────────────────────────────────────────
puntos = alt.Chart(df_sel_con_avg).mark_point(size=40, filled=True, opacity=0.5).encode(
    x="Año:O",
    y="PBI_per_capita:Q",
    color=alt.Color("País:N",
        scale=alt.Scale(domain=list(colores.keys()), range=list(colores.values()))),
    tooltip=[
        alt.Tooltip("País:N",            title="País"),
        alt.Tooltip("Año:O",             title="Año"),
        alt.Tooltip("PBI_per_capita:Q",  title="PBI per cápita (USD)", format="$,.0f")
    ]
)

# ── Etiquetas ──────────────────────────────────────────────────
etiquetas = alt.Chart(df_labels).mark_text(
    align="left", dx=6, fontSize=11, fontWeight="bold"
).encode(
    x="Año:O",
    y="PBI_per_capita:Q",
    text="País:N",
    color=alt.Color("País:N",
        scale=alt.Scale(domain=list(colores.keys()), range=list(colores.values())))
)

# ── Crisis ─────────────────────────────────────────────────────
crisis = pd.DataFrame({
    "Año":    [1989, 2001, 2018],
    "Evento": ["Hiperinflación", "Crisis 2001", "Crisis 2018"]
})
lineas_crisis = alt.Chart(crisis).mark_rule(
    strokeDash=[4,4], strokeWidth=1.2, color="gray", opacity=0.6
).encode(x="Año:O", tooltip=["Evento:N"])

texto_crisis = alt.Chart(crisis).mark_text(
    angle=270, align="right", fontSize=9, color="gray", dy=-5
).encode(x="Año:O", y=alt.value(30), text="Evento:N")

# ── Combinar ───────────────────────────────────────────────────
grafico = (
    lineas_crisis + texto_crisis +
    lineas_paises + linea_arg + linea_avg +
    puntos + etiquetas
).properties(
    title=alt.TitleParams(
        text="PBI per cápita: Argentina vs países seleccionados (1960–2023)",
        subtitle="En dólares corrientes. Línea punteada negra: promedio regional. Líneas grises: crisis argentinas.",
        fontSize=16,
        subtitleFontSize=12,
        anchor="start"
    ),
    width=850,
    height=480
).interactive()

grafico

alt.LayerChart(...)

## 1. Cobertura Temporal

¿Qué años de información podemos extraer? Analizaremos el rango temporal disponible para un indicador global representativo.

In [6]:
def get_time_range(indicator_id="WB_WDI_SP_POP_TOTL", area="WLD"):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator_id, "REF_AREA": area}
    resp = requests.get(url, params=params).json()
    years = [int(v['TIME_PERIOD']) for v in resp['value'] if v['TIME_PERIOD'].isdigit()]
    return min(years), max(years), len(years)

min_y, max_y, total_y = get_time_range()
print(f"Rango temporal disponible: {min_y} - {max_y} ({total_y} años)")

Rango temporal disponible: 1960 - 2024 (65 años)


## 2. Cobertura Geográfica (Países y Áreas)

La API no solo incluye países, sino también agregados regionales (ej. América Latina, Europa) y niveles de ingreso (ej. Países de ingreso bajo).

In [7]:
def get_available_areas(indicator_id="WB_WDI_SP_POP_TOTL"):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator_id, "isLatestData": True}
    resp = requests.get(url, params=params).json()
    areas = [v['REF_AREA'] for v in resp['value']]
    return sorted(list(set(areas)))

areas = get_available_areas()
print(f"Total de áreas/países disponibles: {len(areas)}")
print("Muestra de códigos de área (ISO3):", areas[:25])

Total de áreas/países disponibles: 265
Muestra de códigos de área (ISO3): ['ABW', 'AFE', 'AFG', 'AFW', 'AGO', 'ALB', 'AND', 'ARB', 'ARE', 'ARG', 'ARM', 'ASM', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH']


In [8]:
def get_all_area_metadata():
    """Obtiene metadatos extendidos (nombre, region) de todos los países usando la API de datos abiertos del Banco Mundial"""
    print("Obteniendo metadatos de áreas (regiones, nombres completos)...")
    url = "https://api.worldbank.org/v2/country"
    params = {"format": "json", "per_page": 300}
    try:
        resp = requests.get(url, params=params).json()
        # El formato es [metadata, [datos]]
        countries_data = resp[1]
        metadata_dict = {}
        for c in countries_data:
            metadata_dict[c['id']] = {
                'full_name': c['name'],
                'region': c['region']['value'] if isinstance(c['region'], dict) else 'Unknown',
                'income_level': c['incomeLevel']['value'] if isinstance(c['incomeLevel'], dict) else 'Unknown'
            }
        return metadata_dict
    except Exception as e:
        print(f"Error al obtener metadatos: {e}")
        return {}

area_metadata = get_all_area_metadata()
print(f"Metadatos cargados para {len(area_metadata)} áreas.")

Obteniendo metadatos de áreas (regiones, nombres completos)...
Metadatos cargados para 296 áreas.


## 3. Categorías de Datos (Temas)

Los indicadores están agrupados por temas. Utilizaremos el endpoint de metadatos para descubrir las categorías principales.

In [9]:
def get_topics(top=200):
    url = f"{BASE_URL}/metadata"
    query = {"query": f"&$filter=series_description/database_id eq 'WB_WDI'&$select=series_description/topics/name&$top={top}"}
    resp = requests.post(url, json=query).json()

    unique_topics = set()
    if 'value' in resp:
        for item in resp['value']:
            sd = item.get('series_description')
            if sd is None:
                continue # Skip items where series_description is None

            # La API puede devolver series_description como dict o list segun el dataset
            # Ensure entries contains only dictionaries, filtering out any None or other types
            if isinstance(sd, list):
                entries = [e for e in sd if isinstance(e, dict)]
            elif isinstance(sd, dict):
                entries = [sd]
            else:
                entries = [] # Handle unexpected type for sd, skip if not dict or list

            for entry in entries:
                # entry is guaranteed to be a dict here
                for t in entry.get('topics', []):
                    # Ensure t is a dictionary before calling .get()
                    if isinstance(t, dict) and t.get('name'):
                        unique_topics.add(t['name'])
    return sorted(list(unique_topics))

topics = get_topics()
print(f"Temas principales detectados ({len(topics)}):")
for t in topics: print(f"- {t}")

Temas principales detectados (42):
- Agriculture and Food
- Carbon Pricing, Climate Finance and GHG accounting
- Climate Change
- Connectivity
- Crisis and Disaster Risk Finance
- Digital
- Digital Services
- Economic Policy
- Education
- Energy & Extractives
- Environment
- Finance
- Food and Nutrition Security
- Gender
- Global Infrastructure Finance
- Growth
- Growth and Jobs
- Health, Nutrition & Population
- Housing
- Inequality and Shared Prosperity
- Infrastructure
- Institutions
- Investment and Business Climate
- Jobs
- Learning
- Mortality
- People
- Planet
- Poverty
- Prosperity
- Schooling
- Social Protection
- Strengthening Agi Food Systems
- Teaching
- Trade Outcomes
- Trade, Investment and Competitiveness
- Transmission and Distribution
- Transport
- Trends in the Determinants of Food Security Outcomes
- Urban, Resilience and Land
- Water
- Water and the Economy


## 4. Visualización Exploratoria

### Crecimiento Comparativo en el Cono Sur
Utilizando los datos descubiertos, comparamos el crecimiento poblacional relativo entre países vecinos.

In [10]:
def get_indicator_data(indicator, countries):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator, "REF_AREA": ",".join(countries)}
    resp = requests.get(url, params=params).json()
    df = pd.DataFrame(resp['value'])
    df['OBS_VALUE'] = pd.to_numeric(df['OBS_VALUE'])
    df['TIME_PERIOD'] = pd.to_numeric(df['TIME_PERIOD'])
    return df

df_cono_sur = get_indicator_data("WB_WDI_SP_POP_TOTL", ["ARG", "CHL", "URY", "BRA"])

# Uso de DuckDB para filtrar o procesar si fuera necesario
df_filtered = duckdb.query("SELECT * FROM df_cono_sur WHERE TIME_PERIOD >= 1960").df()

# Visualización interactiva con Altair
line_chart = alt.Chart(df_filtered).mark_line(point=True).encode(
    x=alt.X('TIME_PERIOD:O', title='Año'),
    y=alt.Y('OBS_VALUE:Q', title='Población', scale=alt.Scale(type='log')),
    color=alt.Color('REF_AREA:N', title='País'),
    tooltip=['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']
).properties(
    title='Población Total en el Cono Sur (1960-2023)',
    width=600,
    height=300
).interactive()

line_chart

alt.Chart(...)

### 5. Preparación de Datos para PCA: Agrupando Países por Indicadores

Para realizar un PCA y visualizar países en un plano de dos dimensiones, necesitamos un conjunto de indicadores diversos para varios países. Seleccionaremos algunos indicadores representativos y un grupo más amplio de países para este análisis.

In [11]:
# Definir una lista de indicadores relevantes para PCA
# Usamos algunos de los que ya hemos visto y añadimos otros para mayor diversidad
indicators_for_pca = {
    "WB_WDI_SP_POP_TOTL": "Poblacion Total", # Population, total
    "WB_WDI_NY_GDP_PCAP_CD": "PIB per Capita (USD)", # GDP per capita (current US$)
    "WB_WDI_SP_DYN_LE00_IN": "Esperanza de Vida al Nacer", # Life expectancy at birth, total (years)
    "WB_WDI_SE_ADT_LITR_ZS": "Tasa de Alfabetizacion", # Literacy rate, adult total (% of people ages 15 and above)
    "WB_WDI_EN_ATM_CO2E_PC": "Emisiones CO2 per Capita", # CO2 emissions (metric tons per capita)
    "WB_WDI_SH_MED_BEDS_ZS": "Camas Hospitalarias (per 1k personas)", # Hospital beds (per 1,000 people)
    "WB_WDI_IT_NET_USER_ZS": "Usuarios de Internet (% poblacion)" # Internet users (% of population)
}

# Seleccionar un conjunto de países para el análisis (ej. todos los de América Latina y el Caribe, y algunos de otras regiones)
# Usaremos los códigos ISO3 disponibles en la API
countries_for_pca = areas

# Funcion para obtener datos de un solo indicador para multiples areas y un año específico
def get_indicator_data_for_year(indicator_id, areas, year=None):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator_id, "REF_AREA": ",".join(areas)}
    if year: # Intentar obtener el dato del año especificado, o el más cercano si no hay
        params['TIME_PERIOD'] = str(year)
        # params['isLatestData'] = True # Esto tomaría el último disponible si no hay para el año exacto
    else:
        params['isLatestData'] = True # Si no se especifica año, tomar el más reciente disponible

    resp = requests.get(url, params=params).json()
    df = pd.DataFrame(resp.get('value', []))

    # Asegurarse de que REF_AREA y OBS_VALUE estén presentes y limpios
    if df.empty:
        return pd.DataFrame()

    df['OBS_VALUE'] = pd.to_numeric(df['OBS_VALUE'], errors='coerce')
    df = df.dropna(subset=['OBS_VALUE'])

    # Si hay múltiples entradas por país (ej. diferentes años si no se usó isLatestData o un año específico)
    # nos quedamos con el valor más reciente o el que más se ajuste al año solicitado
    if 'TIME_PERIOD' in df.columns and not year: # Si no se pidió un año específico y se usó isLatestData
        df['TIME_PERIOD'] = pd.to_numeric(df['TIME_PERIOD'], errors='coerce')
        df = df.sort_values(by='TIME_PERIOD', ascending=False).drop_duplicates(subset=['REF_AREA'])
    elif 'TIME_PERIOD' in df.columns and year: # Si se pidió un año específico
        df['TIME_PERIOD'] = pd.to_numeric(df['TIME_PERIOD'], errors='coerce')
        # Intentar obtener el dato del año exacto, si no, el más cercano posterior, luego el más cercano anterior
        df['time_diff'] = abs(df['TIME_PERIOD'] - year)
        df = df.sort_values(by='time_diff').drop_duplicates(subset=['REF_AREA'])

    return df[['REF_AREA', 'OBS_VALUE']].rename(columns={'OBS_VALUE': indicator_id})

# Recopilar los datos para PCA
df_pca_input = pd.DataFrame({'REF_AREA': countries_for_pca})

for indicator_code, indicator_name in indicators_for_pca.items():
    print(f"Obteniendo datos para: {indicator_name} ({indicator_code})...")
    temp_df = get_indicator_data_for_year(indicator_code, countries_for_pca, year=2022) # Intentar obtener datos de 2022
    if not temp_df.empty:
        df_pca_input = pd.merge(df_pca_input, temp_df, on='REF_AREA', how='left')
    else:
        print(f"  No se encontraron datos para {indicator_name} en 2022. Intentando con datos más recientes...")
        temp_df = get_indicator_data_for_year(indicator_code, countries_for_pca) # Obtener el más reciente si 2022 no funciona
        if not temp_df.empty:
             df_pca_input = pd.merge(df_pca_input, temp_df, on='REF_AREA', how='left')
        else:
            print(f"  No se encontraron datos recientes para {indicator_name}.")

# Establecer REF_AREA como índice para facilitar el PCA
df_pca_input = df_pca_input.set_index('REF_AREA')

# Mostrar las primeras filas del DataFrame listo para PCA
print("\nDataFrame preparado para PCA:")
display(df_pca_input.head())

Obteniendo datos para: Poblacion Total (WB_WDI_SP_POP_TOTL)...
Obteniendo datos para: PIB per Capita (USD) (WB_WDI_NY_GDP_PCAP_CD)...
Obteniendo datos para: Esperanza de Vida al Nacer (WB_WDI_SP_DYN_LE00_IN)...
Obteniendo datos para: Tasa de Alfabetizacion (WB_WDI_SE_ADT_LITR_ZS)...
Obteniendo datos para: Emisiones CO2 per Capita (WB_WDI_EN_ATM_CO2E_PC)...
  No se encontraron datos para Emisiones CO2 per Capita en 2022. Intentando con datos más recientes...
  No se encontraron datos recientes para Emisiones CO2 per Capita.
Obteniendo datos para: Camas Hospitalarias (per 1k personas) (WB_WDI_SH_MED_BEDS_ZS)...
Obteniendo datos para: Usuarios de Internet (% poblacion) (WB_WDI_IT_NET_USER_ZS)...

DataFrame preparado para PCA:


,WB_WDI_SP_POP_TOTL,WB_WDI_NY_GDP_PCAP_CD,WB_WDI_SP_DYN_LE00_IN,WB_WDI_SE_ADT_LITR_ZS,WB_WDI_SH_MED_BEDS_ZS,WB_WDI_IT_NET_USER_ZS
REF_AREA,,,,,,
ABW,107310,30975.998912,76.226000,NaN,NaN,NaN
AFE,731821393,1679.327622,64.487020,73.055977,NaN,26.8000
AFG,40578842,357.261153,65.617000,NaN,0.36,17.1917
AFW,497387180,2138.473153,57.987813,60.780979,NaN,37.5000
AGO,35635029,3682.113151,64.246000,NaN,NaN,42.0719


In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import numpy as np

# 1. Preparación y Limpieza de Datos
# Tratar de mantener tantos países como sea posible
df_pca_cleaned = df_pca_input.copy()

# Eliminar indicadores (columnas) con demasiados valores nulos (> 50%)
thresh = len(df_pca_cleaned) * 0.5
df_pca_cleaned = df_pca_cleaned.dropna(axis=1, thresh=thresh)

print(f"Indicadores restantes después de filtrar por completitud: {df_pca_cleaned.shape[1]}")

# Imputar valores faltantes con la media de cada indicador
imputer = SimpleImputer(strategy='mean')
df_imputed = pd.DataFrame(imputer.fit_transform(df_pca_cleaned),
                          columns=df_pca_cleaned.columns,
                          index=df_pca_cleaned.index)

print(f"Realizando PCA sobre {df_imputed.shape[0]} áreas y {df_imputed.shape[1]} indicadores.")

if df_imputed.empty or df_imputed.shape[1] < 2:
    print("No hay suficientes datos para realizar PCA.")
else:
    # 2. Escalado y PCA
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_imputed)

    pca = PCA(n_components=2)
    principal_components = pca.fit_transform(scaled_data)

    pca_results = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'], index=df_imputed.index).reset_index()

    # 3. Enriquecimiento con Metadatos usando DuckDB
    # Convertimos el diccionario de metadatos a DataFrame para DuckDB
    meta_list = [{'REF_AREA': k, **v} for k, v in area_metadata.items()]
    df_meta = pd.DataFrame(meta_list)

    # Unimos resultados de PCA con Metadatos y Datos Originales para Tooltips
    original_data = df_imputed.reset_index()
    # Se filtran las agregaciones regionales (m.region != 'Aggregates') para analizar solo paises
    pca_final = duckdb.query("""
        SELECT
            p.REF_AREA,
            p.PC1,
            p.PC2,
            m.full_name as 'País',
            m.region as 'Región',
            o.* EXCLUDE (REF_AREA)
        FROM pca_results p
        LEFT JOIN df_meta m ON p.REF_AREA = m.REF_AREA
        JOIN original_data o ON p.REF_AREA = o.REF_AREA
        WHERE m.region IS NOT NULL AND m.region != 'Aggregates'
    """).df()

    # 4. Mapeo de nombres legibles para el tooltip
    friendly_tooltips = [
        alt.Tooltip('REF_AREA:N', title='Código ISO'),
        alt.Tooltip('País:N'),
        alt.Tooltip('Región:N'),
        alt.Tooltip('PC1:Q', format='.2f'),
        alt.Tooltip('PC2:Q', format='.2f')
    ]

    # Añadir indicadores originales con nombres legibles al tooltip
    for col in df_imputed.columns:
        friendly_name = indicators_for_pca.get(col, col)
        friendly_tooltips.append(alt.Tooltip(f'{col}:Q', title=friendly_name, format='.2f'))

    # 5. Visualización Interactiva con Altair
    selection = alt.selection_point(fields=['Región'], bind='legend')

    base = alt.Chart(pca_final).encode(
        x=alt.X('PC1:Q', title=f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)'),
        y=alt.Y('PC2:Q', title=f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
    )

    scatter = base.mark_circle(size=80, opacity=0.7).encode(
        color=alt.Color('Región:N', scale=alt.Scale(scheme='tableau10'), title='Región Geográfica'),
        opacity=alt.condition(selection, alt.value(0.9), alt.value(0.1)),
        tooltip=friendly_tooltips
    ).add_params(
        selection
    )

    # Representar codigo ISO del pais en el grafico
    text = base.mark_text(align='left', baseline='middle', dx=7, fontSize=10).encode(
        text='REF_AREA:N',
        color=alt.value('black'),
        opacity=alt.condition(selection, alt.value(0.8), alt.value(0.1))
    )

    chart = (scatter + text).properties(
        title={
            "text": "PCA de Países en el Año 2022 basado en Indicadores de Desarrollo",
            "subtitle": "Interactividad: Clic en la leyenda para filtrar, hover para ver detalles"
        },
        width=800,
        height=600
    ).interactive()

    display(chart)


Indicadores restantes después de filtrar por completitud: 4
Realizando PCA sobre 265 áreas y 4 indicadores.


alt.LayerChart(...)

### Siguientes Pasos: Escalado, PCA y Visualización

Ahora que tenemos el DataFrame `df_pca_input`, los siguientes pasos serían:

1.  **Manejo de Valores Faltantes**: Imputar o eliminar filas/columnas con datos faltantes. (Aunque la función `get_indicator_data_for_year` ya los maneja un poco).
2.  **Escalado de Datos**: Los indicadores tienen diferentes unidades y escalas, por lo que es crucial escalarlos (ej. con `StandardScaler` de `sklearn.preprocessing`) antes de aplicar PCA.
3.  **Aplicar PCA**: Utilizar `PCA` de `sklearn.decomposition` para reducir la dimensionalidad a 2 componentes principales.
4.  **Visualización**: Graficar los países en un diagrama de dispersión usando las dos primeras componentes principales, lo que permitirá identificar grupos de países con características similares.

## Resumen Técnico de la Exploración

- **Rango Temporal**: 1960 a 2024. Ideal para series de largo plazo.
- **Cobertura Espacial**: 265 códigos de área, permitiendo análisis país por país o por bloques económicos (G20, OCDE, etc.).
- **Temáticas**: Diversidad desde agricultura y medio ambiente hasta políticas económicas y agua.
- **Facilidad de Uso**: La API permite filtrado complejo mediante OData (usado en el endpoint de metadatos).

### Interpretación de los Componentes Principales (PCA)

Para entender por qué los países se agrupan de cierta manera en el gráfico interactivo, es fundamental analizar cómo influyen (qué "cargas" o pesos tienen) las variables originales en cada componente principal:

*   **Componente Principal 1 (PC1 - Nivel de Desarrollo Socioeconómico)**:
    Por lo general, el PC1 captura el nivel de desarrollo y riqueza general del país. Variables como el **PIB per Cápita**, la **Esperanza de Vida**, la **Tasa de Alfabetización** y los **Usuarios de Internet** tienen una influencia muy fuerte y positiva en este eje. Por otro lado, suele correlacionarse positivamente con mayores **Emisiones de CO2**. Países situados más a la derecha (PC1 alto) tienden a ser naciones desarrolladas, mientras que los situados a la izquierda enfrentan mayores desafíos socioeconómicos.

*   **Componente Principal 2 (PC2 - Dimensión Demográfica)**:
    El PC2 suele estar fuertemente dominado por variables de escala o tamaño absoluto, principalmente la **Población Total**. Países situados en los extremos de este eje se separan del montón exclusivamente por su masiva cantidad de habitantes (como India o China), diferenciándose independientemente de su nivel de riqueza.

Esta ortogonalidad permite que el gráfico evalúe de forma simultánea el progreso socioeconómico (eje X) y el peso demográfico (eje Y) de cada rincón del mundo.

In [13]:
# 6. Tabla de Datos Tidy, Formateada y Filtrable
from IPython.display import display, HTML
import numpy as np

# Renombrar columnas a nombres legibles
df_tidy = pca_final.rename(columns=indicators_for_pca)

# Mantener solo columnas relevantes y ordenarlas lógicamente
cols_base = ['REF_AREA', 'País', 'Región', 'PC1', 'PC2']
# Extraer indicadores presentes
cols_indicadores = [indicators_for_pca.get(c, c) for c in original_data.columns if c not in ['REF_AREA', 'index']]
columnas_ordenadas = cols_base + [c for c in cols_indicadores if c in df_tidy.columns]
df_tidy = df_tidy[[c for c in columnas_ordenadas if c in df_tidy.columns]]

# Formato personalizado continuo para lograr una tabla 'Tidy'
# - Separador de miles con punto (.)
# - Coma decimal (,)
# - Simplificar > 10.000 como enteros sin decimales
def format_val(x):
    if pd.isna(x):
        return "Sin Datos"
    if isinstance(x, (int, float, np.number)):
        if abs(x) > 10000:
            return f"{x:,.0f}".replace(",", ".")
        else:
            return f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return str(x)

df_formatted = df_tidy.copy()
for col in df_formatted.select_dtypes(include=[np.number]).columns:
    df_formatted[col] = df_formatted[col].apply(format_val)

# Convertir a HTML
html_table = df_formatted.to_html(index=False, table_id="tidy_table", classes="display")

# Añadir inyección para renderizar una tabla de datos altamente interactiva (DataTables JS)
html_code = f"""
<link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
<script type="text/javascript" charset="utf8" src="https://code.jquery.com/jquery-3.7.0.min.js"></script>
<script type="text/javascript" charset="utf8" src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>

{html_table}

<script>
// Evitar colisiones de librerías en múltiples re-ejecuciones
setTimeout(function() {{
    if ($.fn.dataTable.isDataTable('#tidy_table')) {{
        $('#tidy_table').DataTable().destroy();
    }}
    $('#tidy_table').DataTable({{
        "pageLength": 10,
        "lengthMenu": [5, 10, 25, 50, 100],
        "language": {{
            "url": "//cdn.datatables.net/plug-ins/1.13.6/i18n/es-ES.json"
        }}
    }});
}}, 100);
</script>
<style>
/* Estilo limpio (Tidy) para la tabla */
#tidy_table {{
    font-family: Arial, Helvetica, sans-serif;
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
}}
#tidy_table td, #tidy_table th {{
    border: 1px solid #ddd;
    padding: 8px;
    text-align: right;
}}
#tidy_table th {{
    text-align: center;
    background-color: #f2f2f2;
    color: black;
}}
#tidy_table tr:hover {{background-color: #f9f9f9;}}
</style>
"""

display(HTML(html_code))


REF_AREA,País,Región,PC1,PC2,Poblacion Total,PIB per Capita (USD),Esperanza de Vida al Nacer,Usuarios de Internet (% poblacion)
ABW,Aruba,Latin America & Caribbean,"0,52","-0,28",107.310,30.976,"76,23","68,77"
AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan","-2,24","-0,56",40.578.842,"357,26","65,62","17,19"
AGO,Angola,Sub-Saharan Africa,"-1,64","-0,50",35.635.029,"3.682,11","64,25","42,07"
ALB,Albania,Europe & Central Asia,"0,64","-0,16",2.451.636,"7.756,96","78,77","82,61"
AND,Andorra,Europe & Central Asia,"2,02","-0,08",79.705,42.414,"84,02","94,49"
ARE,United Arab Emirates,"Middle East, North Africa, Afghanistan & Pakistan","2,04","-0,12",10.074.977,50.760,"80,49","100,00"
ARG,Argentina,Latin America & Caribbean,"0,67","-0,15",45.407.904,13.962,"75,81","88,38"
ARM,Armenia,Europe & Central Asia,"0,16","-0,24",2.969.200,"6.571,97","74,77","77,03"
ASM,American Samoa,East Asia & Pacific,"0,00","-0,32",48.342,18.017,"72,75","68,77"
ATG,Antigua and Barbuda,Latin America & Caribbean,"0,63","-0,22",92.840,20.105,"77,48","77,16"


## 7. Gráfico de Barras Interactivo · Escala Logarítmica · Filtro por Región

Comparación de indicadores del Banco Mundial entre países de una misma región.  
**Controles:** seleccioná la región geográfica y la variable de interés.  
**Escala:** logarítmica (permite comparar países con órdenes de magnitud muy distintos).  
**Color:** nivel de ingreso (World Bank classification).

In [14]:
import altair as alt
import pandas as pd
import requests

BASE_URL = "https://data360api.worldbank.org/data360"

# ── Indicadores a visualizar ──────────────────────────────────────────────────
INDICATORS = {
    "WB_WDI_NY_GDP_MKTP_CD":    "GDP Total (US$)",
    "WB_WDI_NY_GNP_MKTP_CD":    "GNI Total (US$)",
    "WB_WDI_SP_POP_TOTL":       "Población Total",
    "WB_WDI_SH_XPD_CHEX_PC_CD": "Gasto en Salud per cápita (US$)",
}

# ── Metadatos de países (nombre completo, región, nivel de ingreso) ────────────
def get_country_meta():
    url = "https://api.worldbank.org/v2/country"
    resp = requests.get(url, params={"format": "json", "per_page": 300}).json()
    rows = []
    for c in resp[1]:
        region = c["region"]["value"] if isinstance(c["region"], dict) else None
        if region and region != "Aggregates":
            rows.append({
                "REF_AREA":    c["id"],
                "country_name": c["name"],
                "region":      region,
                "income_level": c["incomeLevel"]["value"]
                               if isinstance(c["incomeLevel"], dict)
                               else "Not classified",
            })
    return pd.DataFrame(rows)

print("Cargando metadatos de países...")
df_meta_bar = get_country_meta()
country_codes_bar = df_meta_bar["REF_AREA"].tolist()
print(f"  {len(country_codes_bar)} países con metadatos")

# ── Función: último dato disponible para un indicador ────────────────────────
def fetch_latest(indicator_id, areas):
    url = f"{BASE_URL}/data"
    params = {
        "DATABASE_ID":  "WB_WDI",
        "INDICATOR":    indicator_id,
        "REF_AREA":     ",".join(areas),
        "isLatestData": "true",   # string lowercase: más compatible con la API
    }
    resp = requests.get(url, params=params).json()
    df = pd.DataFrame(resp.get("value", []))
    if df.empty:
        print(f"    ⚠ Sin datos para {indicator_id}")
        return pd.DataFrame(columns=["REF_AREA", "OBS_VALUE", "TIME_PERIOD"])
    df["OBS_VALUE"]   = pd.to_numeric(df["OBS_VALUE"],   errors="coerce")
    df["TIME_PERIOD"] = pd.to_numeric(df["TIME_PERIOD"], errors="coerce")
    df = df.sort_values("TIME_PERIOD", ascending=False).drop_duplicates("REF_AREA")
    return df[["REF_AREA", "OBS_VALUE", "TIME_PERIOD"]]

# ── Carga de todos los indicadores ────────────────────────────────────────────
frames = []
for ind_id, ind_name in INDICATORS.items():
    print(f"  Cargando: {ind_name}...")
    df_ind = fetch_latest(ind_id, country_codes_bar)
    df_ind["variable"] = ind_name
    frames.append(df_ind)

df_bar = pd.concat(frames, ignore_index=True)
df_bar = df_bar.merge(df_meta_bar, on="REF_AREA", how="left")
df_bar = df_bar.dropna(subset=["OBS_VALUE", "region"])
df_bar["year_label"] = df_bar["TIME_PERIOD"].astype(int).astype(str)

# ── Diagnóstico ───────────────────────────────────────────────────────────────
print(f"\n✓ Dataset listo: {len(df_bar):,} filas · "
      f"{df_bar['REF_AREA'].nunique()} países · "
      f"{df_bar['variable'].nunique()} variables")
print("\nRegiones disponibles:")
for r in sorted(df_bar["region"].unique()):
    n = df_bar[df_bar["region"] == r]["REF_AREA"].nunique()
    print(f"  {r!r}  ({n} países)")
print("\nVariables disponibles:", df_bar["variable"].unique().tolist())
print("\nPrimeras filas:")
display(df_bar.head(4))


Cargando metadatos de países...
  217 países con metadatos
  Cargando: GDP Total (US$)...
  Cargando: GNI Total (US$)...
  Cargando: Población Total...
  Cargando: Gasto en Salud per cápita (US$)...

✓ Dataset listo: 833 filas · 217 países · 4 variables

Regiones disponibles:
  'East Asia & Pacific'  (37 países)
  'Europe & Central Asia'  (58 países)
  'Latin America & Caribbean '  (42 países)
  'Middle East, North Africa, Afghanistan & Pakistan'  (23 países)
  'North America'  (3 países)
  'South Asia'  (6 países)
  'Sub-Saharan Africa '  (48 países)

Variables disponibles: ['GDP Total (US$)', 'GNI Total (US$)', 'Población Total', 'Gasto en Salud per cápita (US$)']

Primeras filas:


,REF_AREA,OBS_VALUE,TIME_PERIOD,variable,country_name,region,income_level,year_label
0,GUY,2.466271e+10,2024,GDP Total (US$),Guyana,Latin America & Caribbean,High income,2024
1,GNB,2.218394e+09,2024,GDP Total (US$),Guinea-Bissau,Sub-Saharan Africa,Low income,2024
2,HTI,2.522415e+10,2024,GDP Total (US$),Haiti,Latin America & Caribbean,Lower middle income,2024
3,HND,3.709357e+10,2024,GDP Total (US$),Honduras,Latin America & Caribbean,Lower middle income,2024


In [15]:
# ── Desactivar límite de filas de Altair (necesario en Colab) ─────────────────
alt.data_transformers.disable_max_rows()

regions_bar   = sorted(df_bar["region"].unique().tolist())
variables_bar = list(INDICATORS.values())

INCOME_ORDER  = [
    "High income", "Upper middle income",
    "Lower middle income", "Low income", "Not classified",
]
INCOME_COLORS = ["#1a7abf", "#4cb4e7", "#f5a623", "#e05c2f", "#aaaaaa"]

# ── Selecciones bound a dropdowns (selection_point es más robusto en Colab) ──
input_region = alt.binding_select(options=regions_bar, name="Región: ")
region_sel = alt.selection_point(
    name="region_sel",
    fields=["region"],
    bind=input_region,
    value="Latin America & Caribbean",
)

input_var = alt.binding_select(options=variables_bar, name="Variable: ")
var_sel = alt.selection_point(
    name="var_sel",
    fields=["variable"],
    bind=input_var,
    value=variables_bar[0],
)

# ── Gráfico principal ─────────────────────────────────────────────────────────
bar_chart = (
    alt.Chart(df_bar)
    .mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4)
    .encode(
        x=alt.X(
            "country_name:N",
            sort=alt.EncodingSortField(field="OBS_VALUE", order="descending"),
            title=None,
            axis=alt.Axis(labelAngle=-50, labelLimit=150, labelFontSize=10),
        ),
        y=alt.Y(
            "OBS_VALUE:Q",
            scale=alt.Scale(type="log"),
            title="Valor (escala logarítmica)",
            axis=alt.Axis(format="~s", grid=True),
        ),
        color=alt.Color(
            "income_level:N",
            title="Nivel de ingreso",
            sort=INCOME_ORDER,
            scale=alt.Scale(domain=INCOME_ORDER, range=INCOME_COLORS),
            legend=alt.Legend(orient="bottom", columns=3),
        ),
        opacity=alt.condition(
            region_sel & var_sel, alt.value(0.85), alt.value(0.0)
        ),
        tooltip=[
            alt.Tooltip("country_name:N",  title="País"),
            alt.Tooltip("region:N",        title="Región"),
            alt.Tooltip("variable:N",      title="Variable"),
            alt.Tooltip("OBS_VALUE:Q",     title="Valor", format=",.2~f"),
            alt.Tooltip("income_level:N",  title="Nivel de ingreso"),
            alt.Tooltip("year_label:N",    title="Año del dato"),
        ],
    )
    .transform_filter(region_sel)
    .transform_filter(var_sel)
    .add_params(region_sel, var_sel)
    .properties(
        title={
            "text": "Indicadores del Banco Mundial por País y Región",
            "subtitle": (
                "Seleccioná región y variable  ·  Escala logarítmica  ·  "
                "Fuente: World Bank Data360 API"
            ),
            "anchor": "start",
            "fontSize": 16,
            "subtitleFontSize": 12,
            "subtitleColor": "#666",
        },
        width=880,
        height=440,
    )
)

bar_chart


alt.Chart(...)